# DINOv3 vs EUPE — STM Modulation Segmentation (SXM raw data)

**Task**: 3-class semantic segmentation

| class_id | class |
|----------|-------|
| 0 | background |
| 1 | modulation_region |
| 2 | sqrt2_modulation_region |

| Encoder | Source | embed_dim | depth | Pretraining |
|---------|--------|-----------|-------|-------------|
| DINOv3 ViT-L | Facebook Research | 1024 | 24 | Natural images |
| EUPE ViT-B | vendor/EUPE | 768 | 12 | Scientific images |

**Pipeline**: .sxm -> detect ROI -> normalize -> Encoder -> Head -> mask
**Annotations**: png-modulation/*.json (LabelMe) -> pct -> SXM pixel via transform_polygon_pct_to_sxm

**Images**: FeTe-sxm/FeTe_*.sxm (raw STM scans)


In [ ]:
from __future__ import annotations
import json

import os, random, sys
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as nn_functional
from torch.utils.data import DataLoader, Dataset, Subset
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image as PILImage, ImageDraw

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for parent in [start, *start.parents]:
        if (parent / 'src' / 'lumen').exists():
            return parent
    raise RuntimeError('Could not find repo root')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
if str(REPO_ROOT / 'hyper-data-main' / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'hyper-data-main' / 'src'))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SXM_DIR = REPO_ROOT / 'data' / 'stm_dataset' / 'FeTe-sxm'
MOD_DIR = SXM_DIR / 'png-modulation'   # LabelMe polygon annotations
SXM_RAW_DIR = SXM_DIR  # .sxm files in root (FeTe_XXXX.sxm)

# ── Encoder checkpoints ──
DINOV3_PATH = REPO_ROOT / 'checkpoints' / 'dinov3-vitl16-pretrain-lvd1689m'
EUPE_PATH   = REPO_ROOT / 'checkpoints' / 'EUPE-ViT-B.pt'

# ── 类别定义 ──
CLASS_NAMES = ['background', 'modulation_region', 'sqrt2_modulation_region']
NUM_CLASSES = len(CLASS_NAMES)
LABEL_TO_CLASS_ID = {
    'modulation_region': 1,
    'sqrt2_modulation_region': 2,
}

IMAGE_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 80
HEAD_LR = 5e-4
ENCODER_LR = 5e-5
WEIGHT_DECAY = 1e-4
DICE_WEIGHT = 0.5
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f'Device: {DEVICE}')
print(f'Repo root: {REPO_ROOT}')
print(f'MOD_DIR exists:         {MOD_DIR.exists()}')
print(f'DINOv3 backbone exists: {DINOV3_PATH.exists()}')
print(f'EUPE backbone exists:   {EUPE_PATH.exists()}')


## 1. SXM + LabelMe Dataset (proper coordinate transform)

Uses `detect_sxm_data_roi` + `transform_polygon_pct_to_sxm` from `sxm_stm.py`.
LabelMe absolute pixel coords -> percentage -> SXM pixel space -> render mask.


In [ ]:
# ── Proper SXM + LabelMe dataset (using sxm_stm transform pipeline) ──
from lumen.data.sxm_stm import (
    detect_sxm_data_roi,
    transform_polygon_pct_to_sxm,
    render_class_mask,
    normalize_sxm_image,
    _ensure_hyperdata_on_path,
)
_ensure_hyperdata_on_path()
from hyperdata.io.sxm_reader import read_sxm_file

class SXMLabelMeSegDataset(Dataset):
    """SXM raw images + LabelMe JSON annotations -> segmentation dataset.

    LabelMe coords are absolute pixels in PNG space.
    We convert: PNG pixels -> percentage -> SXM pixels via the pipeline:
        detect_sxm_data_roi() + transform_polygon_pct_to_sxm()
    Then render masks at SXM resolution with render_class_mask().
    """
    def __init__(self, sxm_dir, anno_dir, label_to_id, image_size=512, augment=False):
        super().__init__()
        self.sxm_dir = Path(sxm_dir)
        self.anno_dir = Path(anno_dir)
        self.label_to_id = label_to_id
        self.image_size = image_size
        self.augment = augment
        self.samples = []
        # Polygon storage (transformed to SXM pixel space), mirrors SXMSegmentationDataset
        self.polys_by_stem: dict[str, list[tuple[list[tuple[float,float]], int]]] = {}
        self.sxm_shape_by_stem: dict[str, tuple[int,int]] = {}
        
        for jf in sorted(self.anno_dir.glob('*.json')):
            sxm_path = self.sxm_dir / (jf.stem + '.sxm')
            png_path = jf.with_suffix('.png')
            if not sxm_path.exists() or not png_path.exists():
                continue
            
            with open(jf, encoding='utf-8') as f:
                ann = json.load(f)
            
            # Filter shapes by label
            shapes = []
            for s in ann.get('shapes', []):
                cid = self.label_to_id.get(s['label'])
                if cid is None: continue
                shapes.append({'label_id': cid, 'points': s['points']})
            if not shapes:
                continue
            
            # Read SXM + detect ROI
            img_raw, meta = read_sxm_file(str(sxm_path))
            roi = detect_sxm_data_roi(str(png_path))

            sxm_rows, sxm_cols = img_raw.shape
            self.sxm_shape_by_stem[jf.stem] = (sxm_rows, sxm_cols)
            
            # PNG dimensions (from annotation metadata)
            png_w = ann.get('imageWidth', 1)
            png_h = ann.get('imageHeight', 1)
            
            # Transform each polygon: LabelMe abs px -> pct -> SXM px
            transformed = []
            for sh in shapes:
                # Convert absolute pixels to percentage (0-100)
                pts_pct = [[float(p[0])/png_w*100, float(p[1])/png_h*100] for p in sh['points']]
                # Transform to SXM pixel space
                pts_sxm = transform_polygon_pct_to_sxm(
                    pts_pct, png_size=(png_w, png_h), roi=roi, sxm_shape=(sxm_rows, sxm_cols)
                )
                transformed.append((pts_sxm, sh['label_id']))
            self.polys_by_stem[jf.stem] = transformed
            self.samples.append({'stem': jf.stem, 'sxm_path': sxm_path})
        
        self.stems = [s['stem'] for s in self.samples]

    def __len__(self):
        return len(self.samples)

    def _load_sxm_and_mask(self, stem):
        sxm_path = self.sxm_dir / (stem + '.sxm')
        img_raw, _meta = read_sxm_file(str(sxm_path))
        sxm_rows, sxm_cols = img_raw.shape
        # Normalize image to [0,1]
        image = normalize_sxm_image(img_raw)
        # Render mask at SXM resolution from pre-transformed polygons
        mask = render_class_mask(self.polys_by_stem[stem], shape=(sxm_rows, sxm_cols))
        return image, mask

    def _maybe_augment(self, image, mask):
        if not self.augment: return image, mask
        if random.random() < 0.5:
            image = image[:, ::-1].copy(); mask = mask[:, ::-1].copy()
        if random.random() < 0.5:
            image = image[::-1, :].copy(); mask = mask[::-1, :].copy()
        k = random.randint(0, 3)
        if k:
            image = np.rot90(image, k).copy(); mask = np.rot90(mask, k).copy()
        if random.random() < 0.5:
            image = np.clip(image * (1.0 + (random.random()-0.5)*0.4) + (random.random()-0.5)*0.2, 0.0, 1.0)
        return image, mask

    def __getitem__(self, index):
        s = self.samples[index]
        image_full, mask_full = self._load_sxm_and_mask(s['stem'])
        S = self.image_size
        # Resize both to target size
        img_pil = PILImage.fromarray((image_full * 255).astype(np.uint8), mode='L')
        img_arr = np.asarray(img_pil.resize((S, S), PILImage.BILINEAR), dtype=np.float32) / 255.0
        mask_pil = PILImage.fromarray(mask_full.astype(np.int32), mode='I')
        mask_arr = np.asarray(mask_pil.resize((S, S), PILImage.NEAREST), dtype=np.int64)
        image_arr, mask_arr = self._maybe_augment(img_arr, mask_arr)
        return {'image': torch.from_numpy(image_arr.astype(np.float32)).unsqueeze(0),
                'mask': torch.from_numpy(mask_arr.astype(np.int64)), 'stem': s['stem']}


dataset = SXMLabelMeSegDataset(
    sxm_dir=SXM_RAW_DIR,
    anno_dir=MOD_DIR,
    label_to_id=LABEL_TO_CLASS_ID,
    image_size=IMAGE_SIZE,
    augment=False,
)
print(f'Classes: {CLASS_NAMES} (num={NUM_CLASSES})')
print(f'Loaded {len(dataset)} SXM+LabelMe samples')
for s in dataset.samples:
    sxm_r, sxm_c = dataset.sxm_shape_by_stem[s['stem']]
    n = len(dataset.polys_by_stem[s['stem']])
    print(f'  {s["stem"]}: SXM={sxm_c}x{sxm_r}  {n} polygons')


## 2. 标注对齐验证（人工检查点）

⚠️ **必须人工肉眼确认**：polygon 是否正确覆盖调制区域。
如果错位明显，说明坐标变换有问题。


In [ ]:
# SXM image is resized to 512x512 (may distort aspect ratio).
# Polygon coords scaled by IMAGE_SIZE / sxm_dims to match.
labeled_indices = [i for i, s in enumerate(dataset.samples) if dataset.polys_by_stem[s['stem']]]
n_show = min(4, len(labeled_indices))
fig, axes = plt.subplots(2, n_show, figsize=(5 * n_show, 10))
if n_show == 1:
    axes = axes[:, np.newaxis]

for col, idx in enumerate(labeled_indices[:n_show]):
    sample = dataset[idx]
    stem = sample['stem']
    img = sample['image'][0].numpy()
    mask = sample['mask'].numpy()
    sxm_r, sxm_c = dataset.sxm_shape_by_stem[stem]
    sx_scale = IMAGE_SIZE / sxm_c
    sy_scale = IMAGE_SIZE / sxm_r
    
    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f"{stem}\nSXM {sxm_c}x{sxm_r}")
    axes[0, col].axis('off')
    
    axes[1, col].imshow(img, cmap='gray')
    # Overlay polygon borders in SXM space (scaled to display)
    for pts, cid in dataset.polys_by_stem[stem]:
        xs = [p[0] * sx_scale for p in pts]
        ys = [p[1] * sy_scale for p in pts]
        c = 'cyan' if cid == 1 else 'magenta'
        axes[1, col].plot(xs + [xs[0]], ys + [ys[0]], color=c, linewidth=1.5)
    axes[1, col].set_title(f'Polygons: {len(dataset.polys_by_stem[stem])}')
    axes[1, col].axis('off')

legend_elements = [
    Line2D([0],[0], color='cyan', linewidth=4, label='modulation_region'),
    Line2D([0],[0], color='magenta', linewidth=4, label='sqrt2_modulation'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2)
plt.tight_layout(); plt.show()
print('Check: polygon borders must align with modulation regions on SXM image.')


## 3. Train/Val Split + Class Weights

4 SXM images -> 3 train / 1 val.
Inverse-frequency class weights to handle background dominance.


In [ ]:
# ── 3 train / 1 val ──
all_indices = list(range(len(dataset)))
random.Random(SEED).shuffle(all_indices)
n_val = max(1, len(all_indices) // 4)
val_indices = sorted(all_indices[:n_val])
train_indices = sorted(all_indices[n_val:])

val_dataset = Subset(dataset, val_indices)
print(f'Train: {len(train_indices)}, Val: {len(val_indices)}')
print(f'Train stems: {[dataset.stems[i] for i in train_indices]}')
print(f'Val stems:   {[dataset.stems[i] for i in val_indices]}')

# ── 训练用增强版本 ──
dataset_aug = SXMLabelMeSegDataset(
    sxm_dir=SXM_RAW_DIR,
    anno_dir=MOD_DIR,
    label_to_id=LABEL_TO_CLASS_ID,
    image_size=IMAGE_SIZE, augment=True,
)
train_dataset = Subset(dataset_aug, train_indices)

# ── 类别像素权重 ──
class_pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in train_indices:
    flat = dataset[i]['mask'].numpy().ravel()
    for cid in range(NUM_CLASSES):
        class_pixel_counts[cid] += int((flat == cid).sum())
freqs = class_pixel_counts / class_pixel_counts.sum()
inv = 1.0 / np.clip(freqs, 1e-6, None)
class_weights = torch.tensor(inv / inv.sum() * NUM_CLASSES, dtype=torch.float32).to(DEVICE)

print('Per-class pixel counts:')
for cid, name in enumerate(CLASS_NAMES):
    print(f'  {cid} ({name}): {class_pixel_counts[cid]} px, weight={class_weights[cid].item():.4f}')

labeled_indices_in_ds = list(range(len(dataset)))


## 4. Build Models

| Trainer | Encoder | Mode | Loss |
|---------|---------|------|------|
| DINOv3 | ViT-L | head_only (frozen) | DiceCELoss |
| EUPE | ViT-B | head_only (frozen) | DiceCELoss |

Both encoders frozen. Only the linear segmentation head is trained.


In [ ]:
from lumen.models import DINOv3Encoder
from lumen.models.eupe import EUPEEncoder
from lumen.training.downstream import SegmentationTrainer

# ── 自定义 CE + Dice 混合 Loss ──
class DiceCELoss(nn.Module):
    """CrossEntropy + Dice loss 混合，对小区域分割更友好。"""
    def __init__(self, ce_weight=None, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=ce_weight, ignore_index=-1)
        self.dice_weight = dice_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        C = logits.shape[1]
        targets_oh = nn_functional.one_hot(targets, C).permute(0, 3, 1, 2).float()
        dice = 0.0; n = 0
        for c in range(1, C):  # skip background (class 0)
            p_c = probs[:, c]; t_c = targets_oh[:, c]
            inter = (p_c * t_c).sum()
            union = p_c.sum() + t_c.sum()
            if union > 0:
                dice += 1.0 - (2.0 * inter + self.smooth) / (union + self.smooth)
                n += 1
        dice_loss = dice / max(n, 1)
        return ce_loss + self.dice_weight * dice_loss

cweight = class_weights.detach().cpu()

# ── DINOv3 trainer (head_only, frozen encoder) ──
print('Loading DINOv3 encoder...')
dino_encoder = DINOv3Encoder(model_dir=DINOV3_PATH, device=DEVICE, local_files_only=True)

dino_trainer = SegmentationTrainer(
    encoder=dino_encoder,
    num_classes=NUM_CLASSES,
    trainability='head_only',
    segmentation_head_name='dinov3-linear',
    segmentation_loss='ce',
    scheduler_name='cosine',
    scheduler_t_max=EPOCHS,
    lr=HEAD_LR,
    weight_decay=WEIGHT_DECAY,
).to(DEVICE)
dino_trainer.criterion = DiceCELoss(ce_weight=cweight, dice_weight=DICE_WEIGHT).to(DEVICE)
d_trainable = sum(p.numel() for p in dino_trainer.parameters() if p.requires_grad)
print(f'  DINOv3: embed_dim={dino_encoder.embed_dim}, depth=24, trainable={d_trainable:,} (head_only)')

# ── EUPE trainer (encoder_and_head, unfrozen) ──
print('Loading EUPE encoder...')
eupe_encoder = EUPEEncoder.from_pretrained(checkpoint_path=EUPE_PATH, device=DEVICE, strict=False)
eupe_encoder.auto_convert_input_channels = True

eupe_trainer = SegmentationTrainer(
    encoder=eupe_encoder,
    num_classes=NUM_CLASSES,
    trainability='head_only',
    segmentation_head_name='dinov3-linear',
    segmentation_loss='ce',
    scheduler_name='cosine',
    scheduler_t_max=EPOCHS,
    lr=HEAD_LR,
    weight_decay=WEIGHT_DECAY,
).to(DEVICE)
eupe_trainer.criterion = DiceCELoss(ce_weight=cweight, dice_weight=DICE_WEIGHT).to(DEVICE)
e_trainable = sum(p.numel() for p in eupe_trainer.parameters() if p.requires_grad)
e_total = sum(p.numel() for p in eupe_trainer.parameters())
print(f'  EUPE: embed_dim={eupe_encoder.embed_dim}, depth=12, trainable={e_trainable:,} (head_only)')

print(f'\nLoss: DiceCELoss(dice_weight={DICE_WEIGHT})')
print(f'DINOv3: head_only  lr={HEAD_LR}')
print(f'EUPE:   head_only  lr={HEAD_LR}')


## 5. 训练循环（DINOv3 head_only / EUPE encoder_and_head，共用 DiceCELoss）


In [ ]:
def collate(batch):
    images = torch.stack([b['image'] for b in batch], dim=0)
    masks = torch.stack([b['mask'] for b in batch], dim=0)
    stems = [b['stem'] for b in batch]
    return {'image': images, 'mask': masks, 'stem': stems}

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate)

@torch.no_grad()
def compute_mean_iou(logits, mask, num_classes):
    pred = logits.argmax(dim=1)
    ious = []
    for c in range(num_classes):
        p = pred == c; g = mask == c
        inter = (p & g).sum().item(); union = (p | g).sum().item()
        if union > 0:
            ious.append(inter / union)
    return float(np.mean(ious)) if ious else 0.0

def train_one_model(name, trainer, encoder):
    history = {'train_loss': [], 'val_loss': [], 'val_miou': []}
    best_miou = -1.0; best_epoch = 0
    CKPT_DIR = REPO_ROOT / 'artifacts' / f'encoder_compare_sxm-mod'
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_CKPT = CKPT_DIR / f'best_{name}.pt'
    print(f'\nTraining: {name}')
    for epoch in range(EPOCHS):
        trainer.train()
        epoch_losses = []
        for batch in train_loader:
            batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
            log = trainer.train_step(batch)
            epoch_losses.append(log['loss'])
        train_loss = float(np.mean(epoch_losses))
        trainer.eval()
        val_losses, val_mious = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
                logits = trainer(batch['image'])
                loss = trainer.compute_loss(logits, batch['mask'])
                val_losses.append(float(loss.item()))
                val_mious.append(compute_mean_iou(logits, batch['mask'], NUM_CLASSES))
        val_loss = float(np.mean(val_losses))
        val_miou = float(np.mean(val_mious))
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        if val_miou > best_miou:
            best_miou = val_miou; best_epoch = epoch
            torch.save({'head_state_dict': trainer.head.state_dict(), 'epoch': epoch, 'val_miou': val_miou}, BEST_CKPT)
        if (epoch + 1) % 20 == 0 or epoch < 5:
            marker = '  [BEST]' if val_miou >= best_miou else ''
            print(f'  [{name}] {epoch+1:3d}/{EPOCHS}: train={train_loss:.3f}, val={val_loss:.3f}, mIoU={val_miou:.3f}{marker}')
    print(f'\nDone. Best mIoU: {best_miou:.4f} (epoch {best_epoch})')
    return history, best_miou, best_epoch, BEST_CKPT

# ── 依次训练 ──
dino_history, dino_best_miou, dino_best_epoch, DINO_BEST = train_one_model(
    'DINOv3_ViT-L', dino_trainer, dino_encoder)
eupe_history, eupe_best_miou, eupe_best_epoch, EUPE_BEST = train_one_model(
    'EUPE_ViT-B', eupe_trainer, eupe_encoder)

print(f'\nSUMMARY:')
print(f'  DINOv3 ViT-L best mIoU: {dino_best_miou:.4f} (epoch {dino_best_epoch})')
print(f'  EUPE   ViT-B best mIoU: {eupe_best_miou:.4f} (epoch {eupe_best_epoch})')
print(f'  Best: {"EUPE" if eupe_best_miou > dino_best_miou else "DINOv3"} wins!')


## 6. 对比可视化：Loss & mIoU 曲线 + 预测


In [ ]:
# ── 训练曲线对比 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_arr = np.arange(1, EPOCHS + 1)

axes[0].plot(epochs_arr, dino_history['train_loss'], 'C0-', alpha=0.4, label='DINOv3 train')
axes[0].plot(epochs_arr, dino_history['val_loss'], 'C0-', linewidth=2, label='DINOv3 val')
axes[0].plot(epochs_arr, eupe_history['train_loss'], 'C1-', alpha=0.4, label='EUPE train')
axes[0].plot(epochs_arr, eupe_history['val_loss'], 'C1-', linewidth=2, label='EUPE val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('DiceCELoss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_arr, dino_history['val_miou'], 'C0-', linewidth=2, label=f'DINOv3 best={dino_best_miou:.4f}')
axes[1].axhline(y=dino_best_miou, color='C0', linestyle='--', alpha=0.5)
axes[1].plot(epochs_arr, eupe_history['val_miou'], 'C1-', linewidth=2, label=f'EUPE best={eupe_best_miou:.4f}')
axes[1].axhline(y=eupe_best_miou, color='C1', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Mean IoU')
axes[1].set_title('Validation mIoU'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
fig.suptitle('DINOv3 ViT-L vs EUPE ViT-B — Training Curves (SXM)', fontsize=13)
plt.tight_layout(); plt.show()

# ── 加载 best heads ──
dino_ckpt = torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)
dino_trainer.head.load_state_dict(dino_ckpt['head_state_dict'])
dino_trainer.eval()
eupe_ckpt = torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)
eupe_trainer.head.load_state_dict(eupe_ckpt['head_state_dict'])
eupe_trainer.eval()

# ── 验证集预测对比 ──
n_show = min(4, len(val_indices))
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

@torch.no_grad()
def _predict_and_ious(trainer, img_t, gt):
    pred = trainer(img_t).argmax(dim=1)[0].cpu().numpy()
    ious = {}
    for c in range(NUM_CLASSES):
        p_c = pred == c; g_c = gt == c
        inter = (p_c & g_c).sum(); union = (p_c | g_c).sum()
        ious[c] = inter / union if union > 0 else float('nan')
    return pred, ious

with torch.no_grad():
    for row in range(min(n_show, len(val_indices))):
        idx = val_indices[row] if row < len(val_indices) else val_indices[0]
        sample = dataset[idx]
        gt = sample['mask'].numpy()
        stem = dataset.stems[idx]
        img_t = sample['image'].unsqueeze(0).to(DEVICE)

        dino_pred, dino_ious = _predict_and_ious(dino_trainer, img_t, gt)
        eupe_pred, eupe_ious = _predict_and_ious(eupe_trainer, img_t, gt)

        axes[row, 0].imshow(sample['image'][0].numpy(), cmap='gray')
        axes[row, 0].set_title(f'{stem}')
        axes[row, 1].imshow(gt, cmap='tab10', vmin=0, vmax=NUM_CLASSES-1)
        axes[row, 1].set_title('Ground Truth')

        d_valid = [v for v in dino_ious.values() if not np.isnan(v)]
        d_iou_str = ', '.join(f'{CLASS_NAMES[c][:4]}={dino_ious[c]:.2f}' for c in range(NUM_CLASSES) if not np.isnan(dino_ious[c]))
        d_overlay = np.zeros((*dino_pred.shape, 4)); d_overlay[dino_pred==1]=[0,1,1,0.35]; d_overlay[dino_pred==2]=[1,0,1,0.35]
        axes[row, 2].imshow(sample['image'][0].numpy(), cmap='gray')
        axes[row, 2].imshow(d_overlay)
        axes[row, 2].set_title(f'DINOv3 (mIoU={np.mean(d_valid):.3f})\n{d_iou_str}')

        e_valid = [v for v in eupe_ious.values() if not np.isnan(v)]
        e_iou_str = ', '.join(f'{CLASS_NAMES[c][:4]}={eupe_ious[c]:.2f}' for c in range(NUM_CLASSES) if not np.isnan(eupe_ious[c]))
        e_overlay = np.zeros((*eupe_pred.shape, 4)); e_overlay[eupe_pred==1]=[0,1,1,0.35]; e_overlay[eupe_pred==2]=[1,0,1,0.35]
        axes[row, 3].imshow(sample['image'][0].numpy(), cmap='gray')
        axes[row, 3].imshow(e_overlay)
        axes[row, 3].set_title(f'EUPE (mIoU={np.mean(e_valid):.3f})\n{e_iou_str}')
        for c in range(4):
            axes[row, c].axis('off')

winner = 'EUPE ViT-B' if eupe_best_miou > dino_best_miou else 'DINOv3 ViT-L'
fig.suptitle(f'Validation Prediction — {winner} wins ({max(dino_best_miou, eupe_best_miou):.4f})', fontsize=14)
plt.tight_layout(); plt.show()

# ── 柱状图对比 ──
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['DINOv3 ViT-L', 'EUPE ViT-B'], [dino_best_miou, eupe_best_miou],
       color=['#2196F3', '#4CAF50'], width=0.5)
ax.set_ylabel('Best Validation mIoU'); ax.set_title('Encoder Comparison (SXM)')
for i, v in enumerate([dino_best_miou, eupe_best_miou]):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylim(0, max(dino_best_miou, eupe_best_miou) * 1.2)
plt.tight_layout(); plt.show()


## 7. Unlabeled SXM Inference

Run inference on .sxm files without matching LabelMe JSONs.


In [ ]:
from hyperdata.io.sxm_reader import read_sxm_file

labeled = {s['stem'] for s in dataset.samples}
unlabeled_sxms = sorted([f for f in SXM_RAW_DIR.glob('*.sxm') if f.stem not in labeled])
print(f'Unlabeled SXM files: {len(unlabeled_sxms)}')

dino_ckpt = torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)
dino_trainer.head.load_state_dict(dino_ckpt['head_state_dict']); dino_trainer.eval()
eupe_ckpt = torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)
eupe_trainer.head.load_state_dict(eupe_ckpt['head_state_dict']); eupe_trainer.eval()

def load_sxm(path, size=IMAGE_SIZE):
    img_raw, _ = read_sxm_file(str(path))
    img_norm = normalize_sxm_image(img_raw)
    img_pil = PILImage.fromarray((img_norm * 255).astype(np.uint8), mode='L')
    img_pil = img_pil.resize((size, size), PILImage.BILINEAR)
    return np.asarray(img_pil, dtype=np.float32) / 255.0

@torch.no_grad()
def infer(trainer, img_np):
    t = torch.from_numpy(img_np.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
    return trainer(t).argmax(dim=1)[0].cpu().numpy()

results = []
for sxm_path in unlabeled_sxms:
    img = load_sxm(sxm_path)
    d_pred = infer(dino_trainer, img)
    e_pred = infer(eupe_trainer, img)
    results.append((sxm_path.stem, img, d_pred, e_pred))

for page, start in enumerate([0, 8]):
    batch = results[start:start+8]
    n = len(batch)
    if n == 0: break
    fig, axes = plt.subplots(n, 3, figsize=(15, 3.5 * n))
    if n == 1: axes = axes.reshape(1, -1)
    for row, (stem, img, dp, ep) in enumerate(batch):
        axes[row, 0].imshow(img, cmap='gray')
        axes[row, 0].set_title(stem, fontsize=9); axes[row, 0].axis('off')
        d_overlay = np.zeros((*dp.shape, 4)); d_overlay[dp==1]=[0,1,1,0.35]; d_overlay[dp==2]=[1,0,1,0.35]
        axes[row, 1].imshow(img, cmap='gray')
        axes[row, 1].imshow(d_overlay)
        axes[row, 1].set_title(f'DINOv3 mod={int((dp==1).sum())} sq2={int((dp==2).sum())}', fontsize=9)
        axes[row, 1].axis('off')
        e_overlay = np.zeros((*ep.shape, 4)); e_overlay[ep==1]=[0,1,1,0.35]; e_overlay[ep==2]=[1,0,1,0.35]
        axes[row, 2].imshow(img, cmap='gray')
        axes[row, 2].imshow(e_overlay)
        axes[row, 2].set_title(f'EUPE mod={int((ep==1).sum())} sq2={int((ep==2).sum())}', fontsize=9)
        axes[row, 2].axis('off')
    fig.suptitle(f'Unlabeled SXM - Page {page+1}/{max(1,(len(results)-1)//8+1)}', fontsize=13)
    plt.tight_layout(); plt.show()

print(f"\n{'stem':12s}  {'DINO mod':>10s}  {'DINO sq2':>10s}  {'EUPE mod':>10s}  {'EUPE sq2':>10s}")
print('-' * 65)
for stem, img, dp, ep in results:
    print(f'{stem:12s}  {int((dp==1).sum()):10d}  {int((dp==2).sum()):10d}  {int((ep==1).sum()):10d}  {int((ep==2).sum()):10d}')


## Summary

**DINOv3 vs EUPE - STM Modulation Segmentation (SXM raw data)**

| Aspect | Detail |
|--------|--------|
| Images | .sxm raw STM scans (1%/99% normalized) |
| Annotations | png-modulation/ 4 LabelMe JSONs |
| Coord transform | LabelMe px -> pct -> SXM px (detect_sxm_data_roi + transform_polygon_pct_to_sxm) |
| Encoders | DINOv3 ViT-L (head_only) vs EUPE ViT-B (encoder_and_head) |
| Head | DINOv3LinearSegmentationHead |
| Loss | DiceCELoss (CE + 0.5xDice) + class weights |


In [ ]:
# Custom pick visualization - edit PICK_STEMS to choose images
# ==========================================================
# Right-click the displayed figure -> "Save Image As" to save manually.
# Or set SAVE_FIG=True below to auto-save a high-res PNG.
PICK_STEMS = ["FeTe_0004", "FeTe_0010", "FeTe_0017", "FeTe_0020"]  # <- change these
SAVE_FIG = False  # True = auto-save high-res PNG to artifacts/custom_pick/

# Load best heads
dino_ckpt = torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)
dino_trainer.head.load_state_dict(dino_ckpt["head_state_dict"]); dino_trainer.eval()
eupe_ckpt = torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)
eupe_trainer.head.load_state_dict(eupe_ckpt["head_state_dict"]); eupe_trainer.eval()

@torch.no_grad()
def load_and_predict_sxm(stem):
    sxm_path = SXM_RAW_DIR / (stem + ".sxm")
    if not sxm_path.exists():
        return None, None, None, None
    img_raw, _ = read_sxm_file(str(sxm_path))
    img_norm = normalize_sxm_image(img_raw)
    img_pil = PILImage.fromarray((img_norm * 255).astype(np.uint8), mode="L")
    img_pil = img_pil.resize((IMAGE_SIZE, IMAGE_SIZE), PILImage.BILINEAR)
    img_np = np.asarray(img_pil, dtype=np.float32) / 255.0
    img_t = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).to(DEVICE)
    d_pred = dino_trainer(img_t).argmax(dim=1)[0].cpu().numpy()
    e_pred = eupe_trainer(img_t).argmax(dim=1)[0].cpu().numpy()
    return img_np, d_pred, e_pred, sxm_path

# Load and display
valid_stems = []
for stem in PICK_STEMS:
    img, dp, ep, sxm_path = load_and_predict_sxm(stem)
    if img is not None:
        valid_stems.append((stem, img, dp, ep))
    else:
        print(f"Warning: {stem}.sxm not found, skipping")

n = len(valid_stems)
if n == 0:
    print("No valid SXM files found for PICK_STEMS")
else:
    fig, axes = plt.subplots(n, 3, figsize=(15, 4.5 * n))
    if n == 1: axes = axes.reshape(1, -1)
    for row, (stem, img, dp, ep) in enumerate(valid_stems):
        has_label = stem in {s["stem"] for s in dataset.samples}
        tag = " [labeled]" if has_label else ""
        # Col 0: input image
        axes[row, 0].imshow(img, cmap="gray")
        axes[row, 0].set_title(f"{stem}{tag}", fontsize=10)
        axes[row, 0].axis("off")
        # Col 1: DINOv3 (transparent overlay)
        axes[row, 1].imshow(img, cmap="gray")
        d_ov = np.zeros((*dp.shape, 4))
        d_ov[dp == 1] = [0, 1, 1, 0.35]
        d_ov[dp == 2] = [1, 0, 1, 0.35]
        axes[row, 1].imshow(d_ov)
        axes[row, 1].set_title(f"DINOv3 mod={int((dp==1).sum())} sq2={int((dp==2).sum())}", fontsize=10)
        axes[row, 1].axis("off")
        # Col 2: EUPE (transparent overlay)
        axes[row, 2].imshow(img, cmap="gray")
        e_ov = np.zeros((*ep.shape, 4))
        e_ov[ep == 1] = [0, 1, 1, 0.35]
        e_ov[ep == 2] = [1, 0, 1, 0.35]
        axes[row, 2].imshow(e_ov)
        axes[row, 2].set_title(f"EUPE mod={int((ep==1).sum())} sq2={int((ep==2).sum())}", fontsize=10)
        axes[row, 2].axis("off")
    plt.tight_layout(pad=1.0)
    # Auto-save (high-res PNG, only if SAVE_FIG=True)
    if SAVE_FIG:
        save_dir = REPO_ROOT / "artifacts" / "custom_pick"
        save_dir.mkdir(parents=True, exist_ok=True)
        fname = save_dir / f"pick_{PICK_STEMS[0]}.png"
        fig.savefig(str(fname), dpi=150, bbox_inches="tight", facecolor="white")
        print(f"Saved: {fname}")
    # Display (right-click figure in Jupyter to Save Image As)
    plt.show()
    # Stats table
    print()
    hdr = "{:12s}  {:>10s}  {:>10s}  {:>10s}  {:>10s}".format(
        "stem", "DINO mod", "DINO sq2", "EUPE mod", "EUPE sq2")
    print(hdr)
    print("-" * 65)
    for stem, img, dp, ep in valid_stems:
        print("{s:12s}  {m1:10d}  {m2:10d}  {e1:10d}  {e2:10d}".format(
            s=stem, m1=int((dp==1).sum()), m2=int((dp==2).sum()),
            e1=int((ep==1).sum()), e2=int((ep==2).sum())))
